In [4]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
langchain-groq \
chromadb \
sentence-transformers \
transformers \
accelerate \
pypdf \
groq

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from transformers import pipeline

from google.colab import files

In [6]:
def load_documents(pdf_paths):

    docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(path)
        docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    return chunks

In [7]:
def create_vectorstore(chunks):

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )

    return vectorstore

In [8]:
from langchain_groq import ChatGroq

def build_rag_chain(vectorstore):
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        api_key="your_groq_api_key_here",
        temperature=0
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    def rag_pipeline(question):
        docs = retriever.invoke(question)
        context = format_docs(docs)
        prompt = f"""You are a World Bank development outcomes analyst.
Use the following excerpts from World Bank project documents to answer the question.
Be specific and cite concrete outcomes, numbers, and findings where available.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:"""
        response = llm.invoke(prompt)
        return {
            "answer": response.content,
            "sources": [doc.metadata.get("source", "unknown") for doc in docs]
        }
    return rag_pipeline


In [9]:
uploaded = files.upload()

pdf_paths = list(uploaded.keys())

Saving Bangladesh Higher Education Acceleration & Transformation — HEAT Project.pdf to Bangladesh Higher Education Acceleration & Transformation — HEAT Project.pdf
Saving Nepal Higher Education Reforms Project — ICR.pdf to Nepal Higher Education Reforms Project — ICR.pdf
Saving Nepal Engineering Education Project — ICR.pdf to Nepal Engineering Education Project — ICR.pdf
Saving Nepal COVID-19 Emergency Response & Health Systems Project (ISR).pdf to Nepal COVID-19 Emergency Response & Health Systems Project (ISR).pdf
Saving Nepal Health Sector ICR (ICR00005984).pdf to Nepal Health Sector ICR (ICR00005984).pdf
Saving Nepal Basic and Primary Education Project II — ICR:PPAR.pdf to Nepal Basic and Primary Education Project II — ICR:PPAR.pdf


In [10]:
chunks = load_documents(pdf_paths)

vectorstore = create_vectorstore(chunks)

chain = build_rag_chain(vectorstore)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
result = chain("What are the project outcomes for Nepal?")

print(result["answer"])

Based on the provided excerpts from the World Bank project documents, the project outcomes for Nepal are as follows:

1. **Decentralization of planning, budgeting, and implementation powers**: The project helped shift planning, budgeting, and implementation powers to district and community levels, which is a significant outcome in line with the government's decentralization strategy in the education sector.

2. **Improved management capacity**: The project aimed to improve management capacity at the central, district, and local levels, which is essential for effective implementation of education policies.

3. **Relevance to human resource development**: The development objectives of the project were highly relevant to the country's human resource development, which is a critical area for Nepal's development.

4. **Co-financing and additional funding**: The project attracted co-financing from donors such as Denmark, Finland, the European Commission, and Norway, totaling US$58.17 million

In [12]:
questions = [
    "What were the key performance indicators and were targets achieved?",
    "What is the overall outcome rating of the project?",
    "What were the achievements of the project development objectives?",
    "What lessons were learned from project implementation?"
]

for q in questions:
    result = chain(q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"Source docs: {result['sources']}")
    print("-" * 50)



Q: What were the key performance indicators and were targets achieved?
A: Based on the provided World Bank project documents, the key performance indicators (KPIs) and their targets for the two projects are as follows:

**India: ICDS Systems Strengthening & Nutrition Improvement Program (ISSNIP)**

1. **Percentage of project districts with district resource groups trained to implement the incremental capacity building system**:
	* Baseline: 62.00%
	* Original Target: 95.00%
	* Revised Target: 100.00%
	* Actual Achieved at Completion: 100.00% (target achieved)
2. **Percentage of targeted Anganwadi Workers (AWWs) trained on the ICT-based management, monitoring, and communication system**:
	* Baseline: 0.00%
	* Original Target: 80.00%
	* Revised Target: 76.94%
	* Actual Achieved at Completion: 76.94% (target achieved)
3. **Percentage of targeted Anganwadi Centers (AWCs) reporting key nutrition and service delivery indicators using ICT systems every month for the last 3 months**:
	* Basel